# Faza 1 — originalni VIPL GRID inference i parity


## Goal

Na istom GRID primeru i istom unseen-speaker checkpoint-u poredi originalni
`model.py` iz pinovanog VIPL commit-a sa minimalno modernizovanim lokalnim modelom.
Kriterijum je: strict load svih težina, isti logits/decode i upstream CER/WER.


## Setup

Izaberi **Runtime → Change runtime type → T4 GPU**. Landmark detekcija i dva
model forward-a namerno se izvršavaju na GPU-u.


In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'face-alignment==1.4.1', 'editdistance>=0.8.1'],
    check=True,
)


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/nikolabakic/Vizuelno-prepoznavanje-govora-na-osnovu-pokreta-usana-pomo-u-LipNet-modela.git'
REPO = Path('/content/lipnet-serbian')
if not (REPO / 'lipnet').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
print('Repo:', REPO)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
import torch

assert torch.cuda.is_available(), 'Uključi T4 GPU u Colab Runtime postavkama.'
DEVICE = torch.device('cuda')
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

UPSTREAM_URL = 'https://github.com/VIPL-Audio-Visual-Speech-Understanding/LipNet-PyTorch.git'
UPSTREAM_SHA = '40209e09c49553c00c25c7d41faa3706aea3c625'
UPSTREAM = Path('/content/VIPL-LipNet-PyTorch')
if not (UPSTREAM / '.git').exists():
    subprocess.run(['git', 'clone', UPSTREAM_URL, str(UPSTREAM)], check=True)
subprocess.run(['git', '-C', str(UPSTREAM), 'checkout', '--detach', UPSTREAM_SHA], check=True)
assert subprocess.check_output(
    ['git', '-C', str(UPSTREAM), 'rev-parse', 'HEAD'], text=True
).strip() == UPSTREAM_SHA


## Steps

### 1. Preuzmi legalni GRID primer, anotaciju i pinovani checkpoint


In [ ]:
from urllib.request import urlretrieve

WORK = Path('/content/faza1')
WORK.mkdir(exist_ok=True)
VIDEO = WORK / 'swwp2s.mpg'
ALIGN = WORK / 'swwp2s.align'
CHECKPOINT = WORK / 'LipNet_unseen_loss_0.44562849402427673_wer_0.1332580699113564_cer_0.06796452465503355.pt'
urls = {
    VIDEO: 'https://spandh.dcs.shef.ac.uk/gridcorpus/examples/id2_vcd_swwp2s.mpg',
    ALIGN: 'https://spandh.dcs.shef.ac.uk/gridcorpus/examples/swwp2s.align',
    CHECKPOINT: (
        'https://raw.githubusercontent.com/VIPL-Audio-Visual-Speech-Understanding/'
        f'LipNet-PyTorch/{UPSTREAM_SHA}/pretrain/LipNet_unseen_loss_0.44562849402427673_wer_0.1332580699113564_cer_0.06796452465503355.pt'
    ),
}
for path, url in urls.items():
    if not path.exists() or path.stat().st_size == 0:
        urlretrieve(url, path)
    print(path.name, f'{path.stat().st_size / 1024**2:.2f} MiB')


### 2. Primeni lokalni kod koji čuva VIPL demo geometriju


In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

from lipnet.demo import (
    make_face_aligner, mouth_tensor, preprocess_video, write_mouth_jpegs,
)

aligner = make_face_aligner(device='cuda', face_detector='sfd')
processed = preprocess_video(VIDEO, aligner, progress_every=15)
video = mouth_tensor(processed.frames).unsqueeze(0)
assert video.shape[1] == 3 and video.shape[-2:] == (64, 128)
assert 0.0 <= float(video.min()) <= float(video.max()) <= 1.0
MOUTH_DIR = WORK / 'mouth_jpegs'
write_mouth_jpegs(processed.frames, MOUTH_DIR)
print('Ulaz (B,C,T,H,W):', tuple(video.shape))
print('Landmark frejmovi:', processed.landmark_frames, '/', processed.decoded_frames)

indices = np.linspace(0, len(processed.frames) - 1, 6, dtype=int)
fig, axes = plt.subplots(2, 3, figsize=(12, 4))
for axis, index in zip(axes.flat, indices):
    axis.imshow(cv2.cvtColor(processed.frames[index], cv2.COLOR_BGR2RGB))
    axis.set_title(f'frame {index}')
    axis.axis('off')
plt.tight_layout()
plt.show()


### 3. Učitaj originalni i lokalni model sa svim checkpoint težinama


In [ ]:
import importlib.util

from lipnet.model import LipNet as LocalLipNet
from lipnet.dataset import MyDataset as LocalDataset
from lipnet.train import load_checkpoint_strict

spec = importlib.util.spec_from_file_location('vipl_original_model', UPSTREAM / 'model.py')
upstream_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(upstream_module)

sys.path.insert(0, str(UPSTREAM))
dataset_spec = importlib.util.spec_from_file_location(
    'vipl_original_dataset', UPSTREAM / 'dataset.py'
)
upstream_dataset_module = importlib.util.module_from_spec(dataset_spec)
dataset_spec.loader.exec_module(upstream_dataset_module)

upstream_dataset = upstream_dataset_module.MyDataset.__new__(
    upstream_dataset_module.MyDataset
)
upstream_frames = upstream_dataset._load_vid(str(MOUTH_DIR))
local_frames = LocalDataset._load_vid(MOUTH_DIR)
np.testing.assert_array_equal(local_frames, upstream_frames)
video = torch.from_numpy(
    np.ascontiguousarray((local_frames / 255.0).transpose(3, 0, 1, 2))
).float().unsqueeze(0)
print('Dataset _load_vid parity:', local_frames.shape)

original_model = upstream_module.LipNet().to(DEVICE)
local_model = LocalLipNet(num_classes=28).to(DEVICE)
original_audit = load_checkpoint_strict(original_model, CHECKPOINT)
local_audit = load_checkpoint_strict(local_model, CHECKPOINT)
assert len(original_audit.loaded) == len(local_audit.loaded)
print('Strict load:', len(local_audit.loaded), 'tenzora; missing=[]; unexpected=[]')


## Checks

### 4. Dokaži parity logits-a, dekodiranog teksta, CER-a i WER-a


In [ ]:
from lipnet.dataset import MyDataset

original_model.eval()
local_model.eval()
with torch.inference_mode():
    original_logits = original_model(video.to(DEVICE))
    local_logits = local_model(video.to(DEVICE))
torch.testing.assert_close(local_logits, original_logits, rtol=1e-6, atol=1e-6)

original_prediction = upstream_dataset_module.MyDataset.ctc_arr2txt(
    original_logits.argmax(-1)[0], start=1
)
local_prediction = MyDataset.ctc_arr2txt(local_logits.argmax(-1)[0], start=1)
assert original_prediction == local_prediction

tokens = []
for line in ALIGN.read_text().splitlines():
    token = line.split()[-1]
    if token.upper() not in {'SIL', 'SP'}:
        tokens.append(token.upper())
truth = ' '.join(tokens)
original_cer = upstream_dataset_module.MyDataset.cer([original_prediction], [truth])[0]
original_wer = upstream_dataset_module.MyDataset.wer([original_prediction], [truth])[0]
cer = MyDataset.cer([local_prediction], [truth])[0]
wer = MyDataset.wer([local_prediction], [truth])[0]
assert cer == original_cer and wer == original_wer
print('Ground truth:', truth)
print('Original   :', original_prediction)
print('Lokalni    :', local_prediction)
print(f'CER={cer:.4f} WER={wer:.4f}')


In [ ]:
import json
result = {
    'phase': 1,
    'upstream_commit': UPSTREAM_SHA,
    'checkpoint_tensors': len(local_audit.loaded),
    'input_shape': list(video.shape),
    'decoded_frames': processed.decoded_frames,
    'landmark_frames': processed.landmark_frames,
    'truth': truth,
    'original_prediction': original_prediction,
    'local_prediction': local_prediction,
    'cer': cer,
    'wer': wer,
}
(WORK / 'phase1_result.json').write_text(
    json.dumps(result, indent=2) + '\n', encoding='utf-8'
)
print(json.dumps(result, indent=2))


## Next Steps

Na Fazu 2 pređi samo ako je strict load potpun, `assert_close` prolazi i oba
modela daju isti tekst. Sačuvaj prikaz mouth ROI-a i `phase1_result.json`.
